# CSCI 4052U — Lab 1
## Shallow Networks, Losses, and Generalization

**Units.** `preliminaries_to_machine_learning`, `training_models`
**Duration.** 1 hour

By the end of this lab you will have, with your own code and your own numbers:

| Exercise | Learning outcome |
|---|---|
| 1 | **`construct-shallow-network`** — construct a shallow ReLU network and trace how it produces a piecewise linear function |
| 2 | **`derive-loss-from-likelihood`** — derive a loss by having the network predict the parameters of a distribution over outputs |
| 3 | **`decompose-test-error`** — separate train from test error and say which of noise, bias, and variance dominates |

### How this notebook works

- Cells marked **SETUP** are provided. Run them; do not change them.
- Cells marked **YOUR CODE** are yours. Each contains `raise NotImplementedError()` — delete that line and write your answer.
- Cells marked **CHECK** verify your work and print `[ok]` when you are right. If a check fails, read its message: it tells you what is wrong, not just that something is.

Support code lives in `lab1_utils.py`. You never need to edit that file.


---
## Setup


In [ ]:
# COLAB SETUP -- run this cell first.  Outside Colab it does nothing.
import importlib.util
import subprocess
import sys
import urllib.request

try:  # find_spec raises when the parent package is absent, so import instead
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab already ships torch and torchvision; install only what is missing.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightning", "torchmetrics"], check=True)

# Fetch the lab's support module when it is not already beside the notebook.
if importlib.util.find_spec("lab1_utils") is None:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/kenpu-uoit/csci4052u-machine-learning-2/main/labs/lab1/lab1_utils.py", "lab1_utils.py")
    print("downloaded lab1_utils.py")


In [ ]:
# SETUP -- run this cell first
# Every library is imported under a name, and its symbols are reached through
# that name: utils.SineDataModule, nn.Linear, F.mse_loss, and so on.
import math

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import lab1_utils as utils

utils.quiet_lightning()
L.seed_everything(0, workers=True)

# The two datasets used in this lab.
sine = utils.SineDataModule(n_train=60, n_test=200, batch_size=16, noise=0.1)
mnist1d = utils.MNIST1DDataModule(batch_size=100)   # downloads once, then cached in ./data

print(f"sine:    {len(sine.x_train)} train, {len(sine.x_test)} test")
print(f"MNIST-1D: {len(mnist1d.x_train)} train, {len(mnist1d.x_test)} test, "
      f"{mnist1d.x_train.shape[1]} inputs, 10 classes")
utils.plot_datasets(sine, mnist1d)


---
# Exercise 1 — A shallow ReLU network, by hand
**Outcome A · about 15 minutes**

The `preliminaries_to_machine_learning` notes define a shallow network with $D$ hidden units as

$$h_d = a[\theta_{d0} + \theta_{d1}x], \qquad y = \phi_0 + \sum_{d=1}^{D}\phi_d h_d,$$

where $a[z] = \mathrm{ReLU}[z] = \max(0, z)$.

Each hidden unit clips a line, and the point at which unit $d$ switches between clipped and
unclipped — its **joint** — is where its line crosses zero:

$$\theta_{d0} + \theta_{d1}x = 0 \quad\Longrightarrow\quad x = -\frac{\theta_{d0}}{\theta_{d1}}.$$

You will work with the exact network from the notes: $D = 3$, with

$$\phi = \{-0.23,\ -1.3,\ 1.3,\ 0.66\}, \qquad
\theta = \{(-0.2, 0.4),\ (-0.9, 0.9),\ (1.1, -0.7)\}.$$

These are already loaded as `utils.EXAMPLE_PHI` (shape `(4,)`, the offset first) and `utils.EXAMPLE_THETA`
(shape `(3, 2)`, each row an intercept and a slope).


### 1a — Implement the network · *coding*

Write `shallow_forward` using **plain tensor arithmetic only**: no `nn.Module`, no `autograd`,
no loops over the data. Broadcasting does the work.

Given `x` of shape `(N,)`, return the network output of shape `(N,)`.

*Hint.* `theta[:, 0] + theta[:, 1] * x.unsqueeze(1)` gives you all `(N, D)` pre-activations at once.


In [ ]:
def shallow_forward(x, theta, phi):
    """Evaluate a shallow ReLU network.

    x     : (N,)     input values
    theta : (D, 2)   row d is (intercept, slope) of hidden unit d
    phi   : (D + 1,) output offset phi_0 followed by the weights phi_1 ... phi_D

    returns: (N,) the network output y
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# CHECK 1a
utils.check_shallow_forward(shallow_forward)


### 1b — See the joints · *visualization and inspection*

Three things to produce:

1. **A plot.** A 2x2 panel over $x \in [0, 2]$: the three hidden activations $h_1, h_2, h_3$ in
   the first three panels, and the network output $y$ in the fourth. On every panel, draw a
   dotted vertical line at each joint.
2. **`joints`** — a tensor of the three joint locations, computed from $\theta$ with the formula above.
3. **`slope`** — the slope of the output on the region $x \in (0.5, 1.0)$, **measured from your
   plotted curve** (a rise-over-run between two points inside the region), not from the formula.

The notes state that on this region $h_1$ and $h_3$ are active while $h_2$ is clipped, so the
slope should come out to $\theta_{11}\phi_1 + \theta_{31}\phi_3$. Your measurement is the check
on that claim.


In [ ]:
x = torch.linspace(0, 2, 2001)

# YOUR CODE HERE
# 1. compute the joints from utils.EXAMPLE_THETA
# 2. measure the slope on the region 0.5 < x < 1.0 from the network output
# 3. draw the 2x2 panel of h_1, h_2, h_3 and y, with the joints marked
raise NotImplementedError()


In [ ]:
# CHECK 1b
utils.check_joints_and_slope(joints, slope)


### 1c — Answer in words · *inspection*

Replace the placeholders below.

1. A network with $D$ hidden units produces at most **`___`** linear regions, because **`___`**.
2. The notes point out that although there are $D + 1$ regions, only $D$ of the region slopes are
   independent. Why? **`___`**
3. If you set $\theta_{21} = 0$ (hidden unit 2 has a flat line), what happens to the number of
   joints, and why? **`___`**


---
# Exercise 2 — The loss recipe
**Outcome B · about 25 minutes**

The `training_models` notes give a four-step recipe that produces *every* loss in this course:

1. Choose a probability distribution $Pr(y|\theta)$ over the output domain, with parameters $\theta$.
2. Set the network to predict those parameters: $\theta = f[x, \phi]$.
3. Find the $\hat\phi$ minimising the **negative log-likelihood** over the training pairs.
4. At test time return the distribution, or the value that maximises it.

You will run the recipe twice. The first time the distribution is a **normal** and you will get
least squares. The second time you change **only step 1** to a **categorical** distribution and
you get cross-entropy. Nothing else about the procedure changes — that is the point of the exercise.


### 2a — Least squares *is* the Gaussian negative log-likelihood · *coding*

Step 1 picks the univariate normal, step 2 sets $\mu = f[x,\phi]$, and step 3 writes

$$L[\phi] = -\sum_{i=1}^{I}\log\left[\frac{1}{\sqrt{2\pi\sigma^2}}\exp\left[-\frac{(y_i - f[x_i,\phi])^2}{2\sigma^2}\right]\right].$$

Implement this **directly from the definition** — take the log of the density and sum it, without
simplifying on paper first. The check will then show you what the algebra in the notes shows: what
you wrote equals $\tfrac{1}{2\sigma^2}\sum_i (y_i - f[x_i,\phi])^2$ plus a constant, so minimising
it and minimising the sum of squares are the same thing.


In [ ]:
def gaussian_nll(pred, y, sigma):
    """Negative log-likelihood of y under N(pred, sigma^2), summed over examples.

    pred  : (I, 1) the network output, used as the mean mu
    y     : (I, 1) the observed targets
    sigma : float, the fixed standard deviation

    returns: a scalar tensor
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# CHECK 2a
utils.check_gaussian_nll(gaussian_nll)


### 2b — Fit the regression model · *coding and visualization*

Now minimise that loss by gradient descent. `utils.LitRegressor` in `lab1_utils.py` supplies
`forward`, `validation_step` and `test_step`; you supply the three pieces that define the model
and its optimisation. Complete them by **subclassing in this notebook** — do not edit `lab1_utils.py`.

- `__init__` — build `self.net` as `nn.Sequential`: `width` is the hidden size, so the shape is
  1 → `width` (ReLU) → `width` (ReLU) → 1.
- `training_step` — compute the mean squared error between `self(x)` and `y`, log it under the
  name `"train_loss"` with `on_epoch=True`, and return it.
- `configure_optimizers` — return `torch.optim.Adam` over `self.parameters()` at `self.hparams.lr`.

Then fit for 200 epochs and plot both the loss curve and the fitted function.


In [ ]:
class MyRegressor(utils.LitRegressor):
    def __init__(self, width=20, lr=0.01):
        super().__init__(width=width, lr=lr)
        # YOUR CODE HERE -- build self.net
        raise NotImplementedError()

    def training_step(self, batch, batch_idx):
        x, y = batch
        # YOUR CODE HERE -- compute the MSE loss, log it as "train_loss", return it
        raise NotImplementedError()

    def configure_optimizers(self):
        # YOUR CODE HERE -- return an Adam optimizer
        raise NotImplementedError()


# fit the model  (the utils.LossHistory callback records the per-epoch loss for you)
history = utils.LossHistory()
model = MyRegressor(width=20, lr=0.01)
trainer = utils.make_trainer(max_epochs=200, quiet=True)
trainer.callbacks.append(history)
trainer.fit(model, datamodule=sine)

# YOUR CODE HERE -- plot history.losses against the epoch number (log-y reads well),
# then call utils.plot_fit(model, sine) to overlay the fitted function on the data


In [ ]:
# CHECK 2b
utils.check_regressor(model, sine, trainer, history.losses)


**Look at your fit before moving on.** It is piecewise linear, exactly like Exercise 1 — but with
20 hidden units per layer instead of 3, the joints are dense enough that the curve looks smooth.
The mechanism has not changed; only the number of pieces has.


### 2c — Change step 1, and only step 1 · *coding*

Now the outputs are class labels $y \in \{0, \dots, 9\}$ rather than real numbers, so step 1 of
the recipe picks the **categorical** distribution instead of the normal. Its $K$ parameters come
from the softmax of $K$ network outputs, and the negative log-likelihood is the **multiclass
cross-entropy** loss.

Everything else is already written for you in `utils.LitClassifier`: the 40 → 100 → 100 → 10 network,
the optimizer, the accuracy metric, `validation_step`, and `test_step`. The **only** thing that
differs from Exercise 2b is the loss function — which is exactly the claim the notes make.

Write `training_step` using `F.cross_entropy`.


In [ ]:
class MyClassifier(utils.LitClassifier):
    def training_step(self, batch, batch_idx):
        x, y = batch
        # YOUR CODE HERE -- cross-entropy of the logits against y, logged as "train_loss"
        raise NotImplementedError()


classifier = MyClassifier(width=100, lr=0.1)
clf_trainer = utils.make_trainer(max_epochs=30, quiet=True)
clf_trainer.fit(classifier, datamodule=mnist1d)
results = clf_trainer.test(classifier, datamodule=mnist1d, verbose=False)
print(results)


In [ ]:
# CHECK 2c
utils.check_classifier(results)


### 2d — Answer in words · *inspection*

1. You changed one line between the regressor and the classifier. Which step of the four-step
   recipe does that line correspond to, and which steps were identical? **`___`**
2. Your MNIST-1D test accuracy is around 60–67%, while the training accuracy is far higher.
   Name that phenomenon. **`___`** (Exercise 3 measures it.)


---
# Exercise 3 — Capacity, train error, and test error
**Outcome C · about 15 minutes**

The `training_models` notes decompose the expected test error of a least-squares model into three
additive parts:

$$\mathbb{E}\big[\text{test error}\big] =
\underbrace{\mathbb{E}_\mathcal{D}\Big[\big(f[x,\phi[\mathcal{D}]] - f_\mu[x]\big)^2\Big]}_{\text{variance}} +
\underbrace{\big(f_\mu[x] - \mu[x]\big)^2}_{\text{bias}} +
\underbrace{\sigma^2}_{\text{noise}}.$$

- **Noise** is irreducible. Our data is generated with $\varepsilon \sim \mathcal{N}(0, 0.25^2)$,
  so **no model can achieve a test MSE below $\sigma^2 = 0.0625$**. You will see this floor in your plot.
- **Bias** is the model being too inflexible to represent the true function. It shows up as
  *high training error*.
- **Variance** is the fit changing with the particular training sample. It shows up as a *gap
  between training and test error*.

To make all three visible you need a small, noisy training set — 15 points with
$\sigma = 0.25$, following the notes, which use fifteen points for exactly this demonstration.


In [ ]:
# SETUP -- a deliberately small and noisy training set
sine_small = utils.SineDataModule(n_train=15, n_test=200, batch_size=8, noise=0.25)
print(f"{len(sine_small.x_train)} training points, noise floor sigma^2 = {sine_small.noise**2:.4f}")


### 3a — Fit at a given capacity · *coding*

Write `fit_and_score(width)`, which trains a fresh `MyRegressor` of the given hidden width on
`sine_small` and returns the tuple `(train_mse, test_mse)`.

Three things to get right:

- Call `L.seed_everything(0, workers=True)` first, so that every width starts from a comparable
  initialization and the sweep is a fair comparison.
- Use `utils.make_trainer(max_epochs=300, quiet=True)` — `quiet=True` suppresses the progress bar, which
  matters once you are fitting seven models.
- Score under `torch.no_grad()`, using `F.mse_loss` against `sine_small.x_train` / `.y_train` and
  `sine_small.x_test` / `.y_test`.


In [ ]:
def fit_and_score(width):
    """Train a MyRegressor of the given hidden width on sine_small.

    returns: (train_mse, test_mse) as plain floats
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# CHECK 3a
utils.check_fit_and_score(fit_and_score)


### 3b — Sweep the capacity · *coding and visualization*

Sweep `widths = [1, 2, 4, 8, 16, 64, 256]`, collecting `train_errors` and `test_errors`, then plot
them with

```python
utils.plot_capacity_sweep(widths, train_errors, test_errors, noise_floor=sine_small.noise**2)
```

The whole sweep takes under ten seconds.


In [ ]:
widths = [1, 2, 4, 8, 16, 64, 256]

# YOUR CODE HERE -- build train_errors and test_errors, then plot the sweep
raise NotImplementedError()


In [ ]:
# CHECK 3b
utils.check_sweep(widths, train_errors, test_errors, noise_floor=sine_small.noise**2)


### 3c — Read the decomposition off your plot · *inspection*

The check above printed the three numbers you need. Fill in the table, then write one sentence
per row naming **bias**, **variance**, or **noise** and saying how you can tell.

| Observation | Value from your run | Which term? | Why |
|---|---|---|---|
| Training error at width 1–2 | `___` | `___` | `___` |
| Gap between train and test error at width 256 | `___` | `___` | `___` |
| The floor that the test error approaches but never crosses | `___` | `___` | `___` |

Then answer:

4. At which width is your test error lowest? **`___`**
5. The classical bias–variance picture predicts test error should rise steadily once capacity
   passes that point. Does yours? Look at your width-256 result and say what you actually observe.
   **`___`**


---
## Done

You have covered three of the twenty-four learning outcomes in these two units:

- **`construct-shallow-network`** — you built a ReLU network from arithmetic, located its joints
  at $-\theta_{d0}/\theta_{d1}$, and confirmed a region's slope is set by which units are active.
- **`derive-loss-from-likelihood`** — you derived least squares from the Gaussian NLL, then
  changed one distribution and got cross-entropy, with everything else identical.
- **`decompose-test-error`** — you measured bias as training error, variance as the train–test
  gap, and saw the noise floor $\sigma^2$ that no model can cross.

### If you finished early

- **Double descent.** Re-run the Exercise 3 sweep on `utils.MNIST1DDataModule(label_noise=0.15)` with
  widths out to 512. The notes predict the test error peaks near the capacity that just memorises
  the training set, and then *descends again*. Do you see it?
- **Learning rate.** Re-fit Exercise 2b at `lr` of $10^{-4}$, $10^{-2}$ and $1.0$. One is too slow,
  one is healthy, one diverges. Which is which, and what does the loss curve look like in each case?
- **Regularization.** Add `weight_decay=1e-2` to the Adam optimizer in your width-256 model. What
  happens to the train–test gap?

These preview `explain-double-descent`, `set-training-hyperparameters`, and
`apply-explicit-regularization`, which the lectures take up next.
